In [ ]:
import sys, os
ROOT = os.path.abspath("..")   # go up one directory from notebooks/
if ROOT not in sys.path:
    sys.path.append(ROOT)

print(ROOT)  


In [ ]:
from src.networks import get_all_networks
from src.config import CONFIG

from src.sampling import (
    sample_domain_points,
    sample_top_surface,
    sample_interface,
    sample_far_field
)

from src.losses import total_loss

from src.pde_residual import (
    residual_layer_FGM,
    residual_halfspace_FGM
)
from src.boundary_conditions import (
    top_surface_bc,
    imperfect_interface_bc,
    halfspace_far_field_bc
)


In [ ]:
import torch
import torch.optim as optim

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


In [ ]:
model_layer, model_half = get_all_networks()

model_layer.to(DEVICE)
model_half.to(DEVICE)


In [ ]:
geom = CONFIG["GEOMETRY"]

# Convert parameter dict values to tensors on DEVICE to preserve
# gradient/device consistency inside residual computations
params_layer_raw = CONFIG["LAYER"]
params_half_raw  = CONFIG["SUBSTRATE"]
import torch
params_layer = {
    kk: torch.tensor(v, device=DEVICE) if not isinstance(v, torch.Tensor) else v.to(DEVICE)
    for kk, v in params_layer_raw.items()
}
params_half = {
    kk: torch.tensor(v, device=DEVICE) if not isinstance(v, torch.Tensor) else v.to(DEVICE)
    for kk, v in params_half_raw.items()
}
dispersion = []   # <-- DEFINE DISPERSION HERE


In [ ]:
c = torch.nn.Parameter(
    torch.tensor(
        (params_layer["mu_0"] / params_layer["rho_0"])**0.5,
        device=DEVICE
    )
)


In [ ]:
optimizer = optim.Adam(
    list(model_layer.parameters()) +
    list(model_half.parameters()) +
    [c],
    lr=1e-2
)


In [ ]:
# Training Loop (Dispersion)
k_values = torch.linspace(
    CONFIG["WAVENUMBER"]["k_min"],
    CONFIG["WAVENUMBER"]["k_max"],
    CONFIG["WAVENUMBER"]["num_k"]
)
k_values

In [ ]:
from sympy import python
import torch
import torch.optim as optim
import numpy as np

from src.networks import get_all_networks
from src.config import CONFIG
from src.sampling import (
    sample_domain_points,
    sample_top_surface,
    sample_interface,
    sample_far_field
)
from src.losses import total_loss

# --------------------------------------------------
# Device
# --------------------------------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --------------------------------------------------
# Geometry and parameters
# --------------------------------------------------
geom = CONFIG["GEOMETRY"]
wavenumber = CONFIG["WAVENUMBER"]

params_layer = CONFIG["LAYER"]
params_half  = CONFIG["SUBSTRATE"]

# Convert to tensors
params_layer = {k: torch.tensor(v, dtype=torch.float32, device=DEVICE) if isinstance(v, (int,float)) else v
                for k,v in params_layer.items()}
params_half  = {k: torch.tensor(v, dtype=torch.float32, device=DEVICE) if isinstance(v, (int,float)) else v
                for k,v in params_half.items()}

# ==================================================
# ARCHITECTURE STUDY
# ==================================================

layers_list  = [3,4,5]
neurons_list = [20,30,40]

architecture_results = []

for depth in layers_list:

    for width in neurons_list:

        print(f"\n{'#'*70}")
        print(f"ARCHITECTURE: depth={depth}, width={width}")
        print(f"{'#'*70}")

        # ==========================================
        # BUILD NETWORKS
        # ==========================================
        model_layer, model_half = get_all_networks(
            width=width,
            depth=depth
        )

        model_layer.to(DEVICE)
        model_half.to(DEVICE)

        dispersion = []

        # ==========================================
        # PHYSICS CHECK
        # ==========================================
        c_shear_layer = torch.sqrt(
            params_layer["mu_0"] / params_layer["rho_0"]
        )

        c_shear_half = torch.sqrt(
            params_half["mu_0"] / params_half["rho_0"]
        )

        print("\n=== LOVE-WAVE PHYSICS CHECK ===")
        print(f"c_shear_layer = {c_shear_layer.item():.3f}")
        print(f"c_shear_half  = {c_shear_half.item():.3f}")

        # ==========================================
        # WAVENUMBER LOOP
        # ==========================================
        k_values = torch.linspace(
            wavenumber["k_min"],
            wavenumber["k_max"],
            wavenumber["num_k"]
        )

        loss_history_all = {}

        for idx, k in enumerate(k_values):

            k = k.to(DEVICE)

            print(f"\n{'='*60}")
            print(f"Training for k = {k.item():.3f}")
            print(f"{'='*60}")

            loss_history_all[k.item()] = {
                "total": [],
                "pde": [],
                "bc": []
            }

            lower = 1.05 * c_shear_layer

            if idx == 0:
                upper = 1.01 * c_shear_half
            else:
                prev_c = torch.tensor(
                    dispersion[-1][1],
                    device=DEVICE
                )

                upper = torch.minimum(
                    1.01 * c_shear_half,
                    prev_c * 0.99
                )

            c_raw = torch.nn.Parameter(
                torch.tensor(1.0, device=DEVICE)
            )

            optimizer_nets = optim.Adam(
                list(model_layer.parameters()) +
                list(model_half.parameters()),
                lr=1e-3
            )

            optimizer_c = optim.Adam([c_raw], lr=1e-4)

            best_loss = float("inf")
            best_c = None
            
            # ======================================
            # EARLY STOPPING SETTINGS
            # ======================================

            patience = 200
            counter = 0

            loss_tolerance = 1e-7



            # ======================================
            # TRAINING LOOP
            # ======================================
           

            for epoch in range(1, 100):

                # ----------------------------------
                # phase velocity
                # ----------------------------------

                c = lower + torch.sigmoid(c_raw) * (
                    upper - lower
                )

                # ----------------------------------
                # collocation points
                # ----------------------------------

                z_layer, z_half = sample_domain_points(
                    2000, geom
                )

                z_top = sample_top_surface(
                    500, geom
                )

                z_int = sample_interface(500)

                z_far = sample_far_field(
                    500, geom
                )

                # ----------------------------------
                # reset gradients
                # ----------------------------------

                optimizer_nets.zero_grad()
                optimizer_c.zero_grad()

                # ----------------------------------
                # compute loss
                # ----------------------------------

                loss, logs = total_loss(

                    model_layer,
                    model_half,

                    z_layer,
                    z_half,

                    z_top,
                    z_int,
                    z_far,

                    params_layer,
                    params_half,

                    k,
                    c,

                    w_pde=1.0,
                    w_bc=5.0,
                    w_int=10.0,
                    w_far=5.0,
                    w_amp=0.5
                )

                logs['bc_total'] = (

                    logs.get('bc_top', 0.0)
                    + logs.get('interface', 0.0)
                    + logs.get('far', 0.0)

                )

                # ----------------------------------
                # backward pass
                # ----------------------------------

                loss.backward()

                # ----------------------------------
                # gradient clipping
                # ----------------------------------

                torch.nn.utils.clip_grad_norm_(

                    list(model_layer.parameters()) +
                    list(model_half.parameters()) +
                    [c_raw],

                    1.0
                )

                # ----------------------------------
                # optimizer step
                # ----------------------------------

                optimizer_nets.step()
                optimizer_c.step()

                # ----------------------------------
                # update best solution
                # ----------------------------------

                if loss.item() < best_loss:

                    best_loss = loss.item()

                    best_c = c.detach().item()

                    counter = 0

                else:

                    counter += 1

                # ----------------------------------
                # stopping criteria
                # ----------------------------------

                if best_loss < loss_tolerance:

                    print(

                        f"Early stopping: "
                        f"loss tolerance reached "
                        f"at epoch {epoch}"

                    )

                    break

                if counter >= patience:

                    print(

                        f"Early stopping: "
                        f"no improvement for "
                        f"{patience} epochs"

                    )

                    break

                # ----------------------------------
                # print progress
                # ----------------------------------

                if epoch % 50 == 0:

                    print(

                        f"Epoch {epoch:4d} | "
                        f"Loss={loss.item():.3e} | "
                        f"PDE={logs['pde']:.2e} | "
                        f"BC={logs['bc_total']:.2e} | "
                        f"c={c.item():.6f}"

                    )

            # ======================================
            # SAVE DISPERSION VALUE
            # ======================================

            dispersion.append([k.item(), best_c])

            print(

                f"✓ Final c(k={k.item():.3f}) = "
                f"{best_c:.6f}"

            )


    
        # ==========================================
        # SAVE DISPERSION CSV
        # ==========================================

        np.savetxt(
            f"PINN_L{depth}_N{width}.csv",
            dispersion,
            delimiter=",",
            header="kH,c",
            comments=""
        )

        print(
            f"Saved CSV: "
            f"PINN_L{depth}_N{width}.csv"
        )

        # ==========================================
        # SAVE MODEL
        # ==========================================

        torch.save({

            "model_layer":
                model_layer.state_dict(),

            "model_half":
                model_half.state_dict(),

            "dispersion":
                dispersion,

            "final_loss":
                best_loss

        },

        f"dispersion_L{depth}_N{width}.pth"

        )

        print(
            f"Saved model: "
            f"dispersion_L{depth}_N{width}.pth"
        )

        # ==========================================
        # STORE RESULTS
        # ==========================================
dispersion_np = np.array(dispersion)

# selected k values
k_targets = [0.10, 0.15, 0.20, 0.25]

result_row = {

    "Depth": depth,
    "Width": width,
    "Final_Loss": best_loss

}

# ------------------------------------------------
# extract c for selected k values
# ------------------------------------------------

for k_target in k_targets:

    idx_best = np.argmin(
        np.abs(dispersion_np[:,0] - k_target)
    )

    c_value = dispersion_np[idx_best,1]

    result_row[f'c(k={k_target})'] = c_value

architecture_results.append(result_row)



# ==============================================
# FINAL SUMMARY TABLE
# ==============================================

import pandas as pd

# ==============================================
# FINAL SUMMARY TABLE
# ==============================================

df = pd.DataFrame(architecture_results)

print("\n")
print("="*120)
print("ARCHITECTURE STUDY SUMMARY")
print("="*120)

print(df)

# ==============================================
# SAVE CSV
# ==============================================

df.to_csv(
    "architecture_study.csv",
    index=False
)

print("\nSaved: architecture_study.csv")




In [ ]:
k0 = list(loss_history_all.keys())[0]

print(len(loss_history_all[k0]["total"]))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))

plt.semilogy(loss_history_all[k0]["total"], label='Total loss')
plt.semilogy(loss_history_all[k0]["pde"], label='PDE loss')
plt.semilogy(loss_history_all[k0]["bc"], label='BC loss')

plt.xlabel('Iterations')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
torch.save({
    "model_layer": model_layer.state_dict(),
    "model_half": model_half.state_dict(),
    "c": c.detach().cpu()
}, "dispersion_pinn.pth")

print("Model saved.")


In [ ]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

L = geom["L"]   # layer thickness
H_trunc = geom["H_trunc"]   # half-space depth



In [ ]:
# z-grid for plotting / post-processing

z_layer = torch.linspace(-L, 0.0, 200).reshape(-1, 1).to(DEVICE)
z_half  = torch.linspace(0.0, H_trunc, 200).reshape(-1, 1).to(DEVICE)



In [ ]:
with torch.no_grad():

    scale = 1e-2
# Layer (complex amplitude)   
    
V_layer = model_layer(z_layer)
V_R = V_layer[:, 0:1]
V_I =  V_layer[:, 1:2]
# Half-space (real amplitude)
V_half =  model_half(z_half)



In [ ]:
import sys, platform
import torch
print("sys.executable:", sys.executable)
print("python:", platform.python_version())
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

In [ ]:
import matplotlib.pyplot as plt
import torch

# dispersion: shape (N, 2) → [k, c]
dispersion = torch.tensor(dispersion)
# dispersion: shape (N, 2) → [k, c]
k_vals = dispersion[:, 0]
c_vals = dispersion[:, 1]


plt.figure(figsize=(6,5))
plt.plot(k_vals, c_vals, 'o-', linewidth=2, markersize=6)

plt.xlabel("Wave number $k$")
plt.ylabel("Phase velocity $c$")
plt.title("Dispersion Relation")
plt.grid(True)

plt.show()



In [ ]:
import matplotlib.pyplot as plt
import torch
import numpy as np

# Your dispersion data
dispersion = torch.tensor(dispersion)
k_vals = dispersion[:, 0]
c_vals = dispersion[:, 1]

# Material properties for layer (from your config)
mu44_layer = 0.3e11 # Pa
rho_layer = 7500.0 # kg/m³
H = 2 # Layer thickness (non-dimensional)

# Calculate reference velocity: √(μ₆₆/ρ) for layer
c_ref_layer = np.sqrt(mu44_layer / rho_layer)# Reference shear speed

print(f"Layer properties:")
print(f"  μ₄₄ = {mu44_layer:.2e} Pa")
print(f"  ρ = {rho_layer:.0f} kg/m³")
print(f"  c_ref = √(μ₄₄/ρ) = {c_ref_layer:.1f} m/s")
print(f"  H = {H}")

# Non-dimensionalize
kH = k_vals * H                     # Dimensionless wavenumber
c_norm = c_vals / c_ref_layer       # Dimensionless phase velocity

# Plot
plt.figure(figsize=(6, 5))
plt.plot(kH, c_norm, 'o-', linewidth=2, markersize=6)

plt.xlabel("Dimensionless wavenumber $kH$")
plt.ylabel("Dimensionless phase velocity $c/\\sqrt{\\mu_{44}/\\rho}$")
plt.title("Dispersion Relation (Non-dimensional)")
plt.grid(True)

plt.show()

# Print the data
print("\nDispersion data (non-dimensional):")
print("   kH        c/c_ref")
for i in range(len(kH)):
    print(f"   {kH[i]:.3f}     {c_norm[i]:.3f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import make_interp_spline

# If not already done, convert tensors to numpy arrays
kH_np = kH.cpu().numpy() if hasattr(kH, 'cpu') else np.array(kH)
c_norm_np = c_norm.cpu().numpy() if hasattr(c_norm, 'cpu') else np.array(c_norm)

# Sort values for interpolation (required for spline)
sort_idx = np.argsort(kH_np)
kH_sorted = kH_np[sort_idx]
c_norm_sorted = c_norm_np[sort_idx]

# Create smooth curve using cubic spline
kH_smooth = np.linspace(kH_sorted.min(), kH_sorted.max(), 300)
spline = make_interp_spline(kH_sorted, c_norm_sorted, k=2)
c_norm_smooth = spline(kH_smooth)

# Plot both discrete points and smooth curve
plt.figure(figsize=(6, 5))
plt.plot(kH_sorted, c_norm_sorted, 'o', label='Discrete points')
plt.plot(kH_smooth, c_norm_smooth, '-', label='Smooth curve', linewidth=2)
plt.xlabel("Dimensionless wavenumber $kH$")
plt.ylabel("Dimensionless phase velocity $c/\\sqrt{\\mu_{44}/\\rho}$")
plt.title("Dispersion Relation (Smooth)")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# Save PINN dispersion data
np.savetxt(
    "PINN_dispersion.csv",
    np.column_stack((kH_sorted, c_norm_sorted)),
    delimiter=",",
    header="kH,c_pinn",
    comments=""
)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import jv, yv
from numpy.lib.scimath import sqrt

# Parameters
rho2 = 2800
rho1 = 7500
L = 2

mu_h = 0.28e11
mu_l = 0.3e11

alpha1 = 0.2
alpha2 = 0.9

s = 100
K = mu_l/(s*L)

P1 = 1e7
P2 = 1e7

beta_l = np.sqrt(mu_l / rho1)
beta_h = np.sqrt(mu_h / rho2)

# Grid (same as MATLAB domain)
x = np.linspace(0.09, 0.5, 400)
y = np.linspace(1.01, 1.6, 400)
X, Y = np.meshgrid(x, y)

# Core expressions
Lambda_l = (X / L)**2 * (Y**2 - 1 - P1 / mu_l)
Lambda_h = sqrt(1 + P2/mu_h - (Y * beta_l / beta_h)**2)

xi0 = sqrt(Lambda_l) / alpha1
xiL = xi0 * (1 - alpha1 * L)

# Bessel
J0_0 = jv(0, xi0)
J1_0 = jv(1, xi0)
Y0_0 = yv(0, xi0)
Y1_0 = yv(1, xi0)

J0_L = jv(0, xiL)
J1_L = jv(1, xiL)
Y0_L = yv(0, xiL)
Y1_L = yv(1, xiL)

# LHS & RHS
denom = (J0_0 * Y1_L - Y0_0 * J1_L)
denom[np.abs(denom) < 1e-10] = np.nan

LHS = (J1_0 * Y1_L - Y1_0 * J1_L) / denom

RHS = (K * mu_h * (alpha2 + (X / L) * Lambda_h)) / (
    mu_l * sqrt(Lambda_l) *
    (mu_h * (alpha2 + (X / L) * Lambda_h) + K)
)

F = np.real(LHS - RHS)

# Plot (exact MATLAB equivalent)
plt.figure(figsize=(7,5))
plt.contour(X, Y, F, levels=[-1e-3, 1e-3], colors='k')
plt.xlabel(r'$kL$')
plt.ylabel(r'$c/\beta_l$')
plt.grid(True)

plt.show()

In [ ]:
print(np.min(F), np.max(F))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import jv, yv
from numpy.lib.scimath import sqrt
from scipy.interpolate import make_interp_spline

# ==============================
# LOAD PINN DATA
# ==============================
pinn = np.loadtxt("PINN_dispersion.csv", delimiter=",", skiprows=1)
kH = pinn[:, 0]
c_pinn = pinn[:, 1]

# ==============================
# PARAMETERS (same as yours)
# ==============================
rho2 = 2800
rho1 = 7500
L = 2

mu_h = 0.28e11
mu_l = 0.3e11

alpha1 = 0.2
alpha2 = 0.9

s = 100
K = mu_l/(s*L)

P1 = 1e7
P2 = 1e7

beta_l = np.sqrt(mu_l / rho1)
beta_h = np.sqrt(mu_h / rho2)

# ==============================
# GRID
# ==============================
x = np.linspace(0.09, 0.5, 400)
y = np.linspace(1.01, 1.45, 400)
X, Y = np.meshgrid(x, y)

# ==============================
# ANALYTICAL FUNCTION
# ==============================
Lambda_l = (X / L)**2 * (Y**2 - 1 - P1 / mu_l)
Lambda_h = sqrt(1 + P2/mu_h - (Y * beta_l / beta_h)**2)

xi0 = sqrt(Lambda_l) / alpha1
xiL = xi0 * (1 - alpha1 * L)

J0_0 = jv(0, xi0)
J1_0 = jv(1, xi0)
Y0_0 = yv(0, xi0)
Y1_0 = yv(1, xi0)

J0_L = jv(0, xiL)
J1_L = jv(1, xiL)
Y0_L = yv(0, xiL)
Y1_L = yv(1, xiL)

denom = (J0_0 * Y1_L - Y0_0 * J1_L)
denom[np.abs(denom) < 1e-10] = np.nan

LHS = (J1_0 * Y1_L - Y1_0 * J1_L) / denom

RHS = (K * mu_h * (alpha2 + (X / L) * Lambda_h)) / (
    mu_l * sqrt(Lambda_l) *
    (mu_h * (alpha2 + (X / L) * Lambda_h) + K)
)

F = np.real(LHS - RHS)

# ==============================
# 🔥 EXTRACT CONTOUR POINTS
# ==============================
cs = plt.contour(X, Y, F, levels=[0])

kH_ana = []
c_ana = []

kH_ana = []
c_ana = []

for seg in cs.allsegs[0]:   # level 0 contour
    kH_ana.extend(seg[:, 0])
    c_ana.extend(seg[:, 1])

        
kH_ana = np.array(kH_ana)
c_ana = np.array(c_ana)

plt.clf()  # clear temporary contour

# ==============================
# SORT analytical
# ==============================
idx = np.argsort(kH_ana)
kH_ana = kH_ana[idx]
c_ana = c_ana[idx]

# ==============================
# SMOOTH PINN
# ==============================
idx_p = np.argsort(kH)
kH_sorted = kH[idx_p]
c_pinn_sorted = c_pinn[idx_p]

kH_smooth = np.linspace(kH_sorted.min(), kH_sorted.max(), 300)
spline = make_interp_spline(kH_sorted, c_pinn_sorted, k=2)
c_pinn_smooth = spline(kH_smooth)

# ==============================
# FINAL PLOT
# ==============================
plt.figure(figsize=(7,5))

# PINN
plt.plot(kH_smooth, c_pinn_smooth, '--', linewidth=3, label='PINN')
plt.plot(kH, c_pinn, 's', markersize=5)

# Analytical
plt.plot(kH_ana, c_ana, 'o',
         markerfacecolor='none',
         markersize=2, label='Analytical')

plt.xlabel(r'$kL$', fontsize=12)
plt.ylabel(r'$c/\beta^{(l)}$', fontsize=12)

plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(frameon=False)

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================================
# SAVE ANALYTICAL DISPERSION CURVE
# ==========================================================

analytical_output = np.column_stack(
    (kH_ana, c_ana)
)

np.savetxt(
    "Analytical_dispersion.csv",
    analytical_output,
    delimiter=",",
    header="kH,c",
    comments=""
)

print("\nSaved: Analytical_dispersion.csv")



In [ ]:
# ==============================
# ERROR COMPUTATION (CORRECT WAY)
# ==============================

import numpy as np

# ---- Ensure analytical data is sorted ----
idx = np.argsort(kH_ana)
kH_ana = kH_ana[idx]
c_ana = c_ana[idx]

# ---- Interpolate analytical solution onto PINN kH ----
c_ana_interp = np.interp(kH, kH_ana, c_ana)

# ---- Compute errors ----
abs_error = np.abs(c_pinn - c_ana_interp)
rel_error = abs_error / np.abs(c_ana_interp)

# ==============================
# PRINT TABLE
# ==============================
print("\nCOMPARISON TABLE")
print("kH        PINN c/cs   Analytical c/cs   Abs. Error   Rel. Error")

for i in range(len(kH)):
    print(f"{kH[i]:.4f}   {c_pinn[i]:.6f}   {c_ana_interp[i]:.6f}   {abs_error[i]:.3e}   {rel_error[i]:.3e}")

# ==============================
# SAVE CSV (FOR PAPER)
# ==============================
table = np.column_stack((kH, c_pinn, c_ana_interp, abs_error, rel_error))

np.savetxt(
    "comparison_table.csv",
    table,
    delimiter=",",
    header="kH,PINN_c/cs,Analytical_c/cs,Abs_Error,Rel_Error",
    comments=""
)

print("\nSaved: comparison_table.csv")

# ==============================
# ERROR METRICS (IMPORTANT FOR PAPER)
# ==============================
rmse = np.sqrt(np.mean(abs_error**2))
mae = np.mean(abs_error)
max_err = np.max(abs_error)

print("\nError Metrics:")
print(f"RMSE = {rmse:.4e}")
print(f"MAE  = {mae:.4e}")
print(f"Max Error = {max_err:.4e}")

In [ ]:
plt.figure(figsize=(7,4))

plt.plot(kH, abs_error, 'o-', linewidth=2, markersize=4)
plt.xlabel(r'$kH$', fontsize=12)
plt.ylabel('Absolute Error', fontsize=12)

plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# ==============================
# Absolute error contour (fixed)
# ==============================

# artificial vertical axis
y = np.linspace(0, 1, 50)

# create meshgrid using PINN kH
KH, Y = np.meshgrid(kH, y)

# repeat error along vertical direction
Z = np.tile(abs_error, (len(y), 1))

# ==============================
# PLOT
# ==============================
plt.figure(figsize=(7,4))

cont = plt.contourf(KH, Y, Z, levels=40, cmap='plasma')
cbar = plt.colorbar(cont)
cbar.set_label('Absolute Error')

plt.xlabel(r'$kH$', fontsize=12)
plt.ylabel('', fontsize=10)
plt.yticks([])

plt.xlim([min(kH), max(kH)])
plt.grid(True, linestyle=':', alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# ==============================
# PREPARE CORRECT ARRAYS
# ==============================
# ensure analytical is sorted
idx = np.argsort(kH_ana)
kH_ana = kH_ana[idx]
c_ana = c_ana[idx]

# interpolate analytical onto PINN kH
c_ana_interp = np.interp(kH, kH_ana, c_ana)

pred = np.asarray(c_pinn).flatten()
true = np.asarray(c_ana_interp).flatten()

# ==============================
# ERROR COMPUTATION
# ==============================
err = pred - true
abs_err = np.abs(err)

L1   = np.mean(abs_err)
RMSE = np.sqrt(np.mean(err**2))
L2   = np.sqrt(np.sum(err**2))
Linf = np.max(abs_err)
relL2 = L2 / np.sqrt(np.sum(true**2))

labels = ['L1', 'RMSE', 'L2 norm', 'L∞', 'Rel L2']
values = [L1, RMSE, L2, Linf, relL2]

In [ ]:
plt.figure(figsize=(6,4))

colors = ['#4C72B0','#55A868','#C44E52','#8172B2','#CCB974']
bars = plt.bar(labels, values, color=colors, edgecolor='black', linewidth=0.8)

plt.yscale('log')
plt.ylabel('Error value', fontsize=12)

# value labels (avoid overlap in log scale)
for b in bars:
    h = b.get_height()
    plt.text(b.get_x()+b.get_width()/2, h*1,
             f'{h:.1e}',
             ha='center', va='bottom', fontsize=9)

plt.grid(True, linestyle=':', axis='y', alpha=0.6)
plt.xticks(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
import os

# ==========================================================
# ARCHITECTURES
# ==========================================================

layers_list  = [3,4,5]
neurons_list = [20,30,40]

# ==========================================================
# LOAD ANALYTICAL DATA
# ==========================================================

# your analytical dispersion CSV
analytical = np.loadtxt(
    "Analytical_dispersion.csv",
    delimiter=",",
    skiprows=1
)

kH_ana = analytical[:,0]
c_ana  = analytical[:,1]

# sort
idx = np.argsort(kH_ana)

kH_ana = kH_ana[idx]
c_ana  = c_ana[idx]

# ==========================================================
# NORMALIZATION
# ==========================================================

rho1 = 7500
mu_l = 0.3e11

beta_l = np.sqrt(mu_l / rho1)



# interpolation function
analytical_interp = interp1d(
    kH_ana,
    c_ana,
    bounds_error=False,
    fill_value="extrapolate"
)

# ==========================================================
# SELECTED k VALUES FOR COMPARISON
# ==========================================================


k_targets = [0.105, 0.144, 0.197, 0.250]



# ==========================================================
# FINAL RESULTS TABLE
# ==========================================================

final_table = []

# ==========================================================
# LOOP OVER ARCHITECTURES
# ==========================================================

for depth in layers_list:

    for width in neurons_list:

        filename = f"PINN_L{depth}_N{width}.csv"

        # --------------------------------------------------
        # check file exists
        # --------------------------------------------------

        if not os.path.exists(filename):

            print(f"Missing file: {filename}")
            continue

        print(f"\nReading: {filename}")

        # --------------------------------------------------
        # LOAD PINN DATA
        # --------------------------------------------------

        pinn = np.loadtxt(
            filename,
            delimiter=",",
            skiprows=1
        )

        # convert PINN k -> kL
        L = 2

        kH = pinn[:,0] * L


        c_pinn = pinn[:,1]

        # --------------------------------------------------
        # SORT
        # --------------------------------------------------

        idx_p = np.argsort(kH)

        kH = kH[idx_p]
        c_pinn = c_pinn[idx_p]

        # --------------------------------------------------
        # CREATE ROW
        # --------------------------------------------------

        row = {

            "Depth": depth,
            "Width": width

        }

        errors = []

        # --------------------------------------------------
        # COMPARE MULTIPLE k VALUES
        # --------------------------------------------------

        for k_target in k_targets:

            # nearest PINN point
            idx_best = np.argmin(
                np.abs(kH - k_target)
            )

            k_used = kH[idx_best]

            
            # normalize PINN prediction
            c_pred = c_pinn[idx_best] / beta_l

            print(
                f"Requested k = {k_target:.3f} | "
                f"Used k = {k_used:.3f}"
            )



            # analytical value
            c_exact = analytical_interp(k_used)

            # relative error
            rel_error = abs(
                c_pred - c_exact
            ) / abs(c_exact)

            errors.append(rel_error)

            # store values
            row[f"k={k_target:.2f}_PINN"] = c_pred

            row[f"k={k_target:.2f}_ANA"] = c_exact

            row[f"k={k_target:.2f}_ERR"] = rel_error

        # --------------------------------------------------
        # GLOBAL METRICS
        # --------------------------------------------------

        row["Mean_Error"] = np.mean(errors)

        row["Max_Error"] = np.max(errors)

        final_table.append(row)

# ==========================================================
# CREATE DATAFRAME
# ==========================================================

df = pd.DataFrame(final_table)

# ==========================================================
# SORT BY MEAN ERROR
# ==========================================================

df = df.sort_values("Mean_Error")

# ==========================================================
# PRINT TABLE
# ==========================================================

print("\n")
print("="*150)
print("FINAL ARCHITECTURE ANALYSIS")
print("="*150)

print(df.to_string(index=False))

# ==========================================================
# SAVE CSV
# ==========================================================

df.to_csv(
    "final_architecture_analysis.csv",
    index=False
)

print("\nSaved: final_architecture_analysis.csv")

# ==========================================================
# BEST ARCHITECTURE
# ==========================================================

best = df.iloc[0]

print("\n")
print("="*60)
print("BEST ARCHITECTURE")
print("="*60)

print(f"Depth       : {best['Depth']}")
print(f"Width       : {best['Width']}")
print(f"Mean Error  : {best['Mean_Error']:.3e}")
print(f"Max Error   : {best['Max_Error']:.3e}")

# ==========================================================
# CREATE HEATMAP
# ==========================================================

heatmap = np.zeros(
    (len(layers_list), len(neurons_list))
)

for i, depth in enumerate(layers_list):

    for j, width in enumerate(neurons_list):

        subset = df[
            (df["Depth"] == depth)
            &
            (df["Width"] == width)
        ]

        if len(subset) > 0:

            heatmap[i,j] = subset[
                "Mean_Error"
            ].values[0]

        else:

            heatmap[i,j] = np.nan

# ==========================================================
# PLOT HEATMAP
# ==========================================================

plt.figure(figsize=(7,5))

im = plt.imshow(
    heatmap,
    aspect='auto',
    cmap='viridis'
)

plt.colorbar(
    im,
    label='Mean Relative Error'
)

plt.xticks(
    range(len(neurons_list)),
    neurons_list
)

plt.yticks(
    range(len(layers_list)),
    layers_list
)

plt.xlabel("Neurons per Layer")
plt.ylabel("Hidden Layers")

# annotate cells
for i in range(len(layers_list)):

    for j in range(len(neurons_list)):

        plt.text(
            j,
            i,
            f"{heatmap[i,j]:.1e}",
            ha='center',
            va='center',
            color='white',
            fontsize=9
        )

plt.tight_layout()

plt.show()

